# Tutorial: Evo2 Phage Replication Walkthrough

Audience:
- Engineers or agents reproducing Arc's Evo2 Microviridae phage-design workflow in BioNeMo.
- Readers who want a command-by-command checkpoint, QC, reward, and RL scaffold overview.

Prerequisites:
- Run from the repository root.
- Build the recipe environment with `recipes/evo2_phage_gen/.ci_build.sh` and source `recipes/evo2_phage_gen/.ci_test_env.sh` before running shell commands manually. The build script repairs upstream NeMo-RL package discovery and applies the Evo2/MBridge patch.
- Prepare external reference data under `recipes/evo2_phage_gen/data/external` with `evo2_phage_prepare_external_assets`.

Learning goals:
- Validate the Vortex-to-MBridge converter with the CI-friendly 1B checkpoint test.
- Convert Arc's Microviridae Vortex checkpoint to MBridge and verify exact export back to Vortex.
- Reproduce the dependency-light nucleotide QC and online reward smoke checks.
- Understand how the NeMo-RL GRPO scaffold runs Evo2 generation with Megatron CUDA graphs.


## Outline

1. Define paths and lightweight helpers.
2. Validate the converter with the CI-friendly 1B Vortex checkpoint.
3. Convert and verify the Microviridae checkpoint for baseline generation.
4. Run nucleotide QC and online reward smoke checks on Arc's generated FASTA.
5. Inspect the NeMo-RL GRPO scaffold and current integration gap.
6. Exercises and extension points.


In [ ]:
from __future__ import annotations

import csv
import subprocess
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Find this checkout from the current notebook or shell working directory."""
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "recipes" / "evo2_phage_gen").exists() and (path / "recipes" / "evo2_megatron").exists():
            return path
    raise RuntimeError(f"Could not find repository root from {start}")


REPO = find_repo_root()
RECIPE_REL = Path("recipes/evo2_phage_gen")
RECIPE = REPO / RECIPE_REL
CHECKPOINTS_REL = RECIPE_REL / "data" / "checkpoints"
CHECKPOINTS = REPO / CHECKPOINTS_REL
ARC_PHAGE = RECIPE / "data" / "external" / "arc_evo2" / "phage_gen"

# Keep notebook execution lightweight by default. Flip this to True only when
# you intentionally want to download/convert multi-GB checkpoints from a notebook.
RUN_HEAVY = False

for path in [RECIPE, ARC_PHAGE, CHECKPOINTS]:
    print(path.relative_to(REPO), "exists=", path.exists())

## Step 1 - CI-Friendly Converter Validation

The converter test downloads the smaller public 1B Vortex checkpoint from `arcinstitute/evo2_1b_base`, converts it to MBridge state-dict form, converts back to Vortex, and asserts exact key/value equality. This is the checkpoint round-trip that should be allowed to run in CI.

Run manually from the recipe environment:

```bash
cd recipes/evo2_phage_gen
source .ci_test_env.sh
cd ../..
EVO2_CHECKPOINT_CACHE_DIR=recipes/evo2_phage_gen/data/checkpoints \
python -m pytest \
  recipes/evo2_megatron/tests/bionemo/evo2/utils/checkpoint/test_vortex_to_mbridge.py \
  -q
```

Expected result: `5 passed`.


In [ ]:
converter_test_cmd = [
    "bash",
    "-lc",
    "source recipes/evo2_phage_gen/.ci_test_env.sh && "
    "EVO2_CHECKPOINT_CACHE_DIR=recipes/evo2_phage_gen/data/checkpoints "
    "python -m pytest recipes/evo2_megatron/tests/bionemo/evo2/utils/checkpoint/test_vortex_to_mbridge.py -q",
]

if RUN_HEAVY:
    subprocess.run(converter_test_cmd, cwd=REPO, check=True)
else:
    print("RUN_HEAVY is false; command to run manually:")
    print(converter_test_cmd[-1])

## Step 2 - Convert The Microviridae Checkpoint

The scientific replication starts from Arc's released 7B Microviridae Vortex checkpoint, but this is an operational bootstrap rather than the CI test because the checkpoint is large. Save a local copy first so exact comparison has a stable original path.

```bash
source recipes/evo2_phage_gen/.ci_test_env.sh
mkdir -p recipes/evo2_phage_gen/data/checkpoints
python - <<'PY'
from pathlib import Path
import shutil

from huggingface_hub import hf_hub_download

src = hf_hub_download(
    repo_id="evo-design/evo-2-7b-8k-microviridae",
    filename="evo2_7b_microviridae.pt",
)
dst = Path("recipes/evo2_phage_gen/data/checkpoints/evo2_7b_microviridae.pt")
shutil.copy2(src, dst)
print(dst)
PY

evo2_convert_vortex_to_mbridge \
  --vortex-ckpt-path recipes/evo2_phage_gen/data/checkpoints/evo2_7b_microviridae.pt \
  --mbridge-ckpt-dir recipes/evo2_phage_gen/data/checkpoints/evo2_7b_microviridae_mbridge \
  --model-size evo2_7b_base \
  --seq-length 10240 \
  --tokenizer-path recipes/evo2_phage_gen/tokenizers/nucleotide_fast_tokenizer_512

evo2_export_mbridge_to_vortex \
  --mbridge-ckpt-dir recipes/evo2_phage_gen/data/checkpoints/evo2_7b_microviridae_mbridge \
  --output-path recipes/evo2_phage_gen/data/checkpoints/converted_evo2_7b_microviridae.pt \
  --model-size evo2_7b_base
```

Expected exact validation: 386 original keys, 386 converted keys, zero missing/extra keys, exact tensor or byte equality for every value.


In [ ]:
exact_compare_code = r"""
from io import BytesIO
from pathlib import Path
import torch
from torch.serialization import safe_globals

orig_path = Path('recipes/evo2_phage_gen/data/checkpoints/evo2_7b_microviridae.pt')
conv_path = Path('recipes/evo2_phage_gen/data/checkpoints/converted_evo2_7b_microviridae.pt')
with safe_globals([BytesIO]):
    orig = torch.load(orig_path, map_location='cpu', weights_only=True, mmap=True)
    conv = torch.load(conv_path, map_location='cpu', weights_only=True, mmap=True)
missing = set(orig) - set(conv)
extra = set(conv) - set(orig)
print(f'original_keys={len(orig)} converted_keys={len(conv)} missing={len(missing)} extra={len(extra)}')
assert not missing and not extra
for key in sorted(orig):
    a = orig[key]
    b = conv[key]
    if isinstance(a, BytesIO):
        assert isinstance(b, BytesIO), key
        assert a.getvalue() == b.getvalue(), key
    else:
        assert torch.equal(a, b), key
print('EXACT_ROUNDTRIP_OK')
"""

if RUN_HEAVY:
    subprocess.run(
        ["bash", "-lc", f"source {RECIPE_REL}/.ci_test_env.sh && python - <<'PY'\n{exact_compare_code}\nPY"],
        cwd=REPO,
        check=True,
    )
else:
    print("RUN_HEAVY is false; exact comparison code is available in `exact_compare_code`.")

## Step 3 - Paper-Style Prompt Sweep

For the targeted Microviridae design stage, the paper used prefixes of the PhiX174-like consensus start rather than taxonomy prompts. The released PhiX174-variant FASTA and bundled PhiX174 reference share the same start, `GAGTTTTAT...`. Methods B.1.13-B.1.14 state that prompt lengths spanned 1-11 nucleotides, all prompts were prepended with the `+~` fine-tuning tokens, temperatures were swept across 0.3, 0.5, 0.7, 0.9, and 1.1, and sampling used top-k 4 and top-p 1 with about 1000 sequences per temperature/prompt combination. The useful regime was 4-9 nt prompts and temperature 0.7-0.9.

The first-pass prompt set for this recipe is:

| Prompt length | Prompt |
| --- | --- |
| 4 | `+~GAGT` |
| 5 | `+~GAGTT` |
| 6 | `+~GAGTTT` |
| 7 | `+~GAGTTTT` |
| 8 | `+~GAGTTTTA` |
| 9 | `+~GAGTTTTAT` |

The raw `infer_evo2` JSONL has separate `prompt` and `completion` fields. FASTA reconstruction strips prompt soft tokens such as `+~`, prepends only the nucleotide prefix, and leaves non-DNA completion tokens for QC to reject.

In [ ]:
reference_start = "GAGTTTTATCGCTTCCATGACGCAGAAGTTAACACTTTCGGATATTTCTGATGAGTCGAA"
prompts = {n: f"+~{reference_start[:n]}" for n in range(4, 10)}
prompts

In [ ]:
prompt_sweep_example = """
mkdir -p recipes/evo2_phage_gen/data/checkpoints/generation/prompts
RL_CHECKPOINT=recipes/evo2_phage_gen/data/checkpoints/evo2_7b_microviridae_mbridge
VLLM_EXPORT=recipes/evo2_phage_gen/data/checkpoints/evo2_7b_microviridae_vllm
TOKENIZER_JSON=recipes/evo2_megatron/tokenizers/nucleotide_fast_tokenizer_512/tokenizer.json

evo2_phage_generation write-prompts \\
  --output-dir recipes/evo2_phage_gen/data/checkpoints/generation/prompts \\
  --prompt-lengths 4 \\
  --num-prompts 1000 \\
  --id-prefix phix174

evo2_export_mbridge_to_vllm "$RL_CHECKPOINT" "$VLLM_EXPORT" --max-shard-size 2GiB

infer_evo2 \\
  --model "$VLLM_EXPORT" \\
  --rl-checkpoint "$RL_CHECKPOINT" \\
  --rl-tokenizer-json "$TOKENIZER_JSON" \\
  --prompt-file recipes/evo2_phage_gen/data/checkpoints/generation/prompts/phix174_prompt4_1000.jsonl \\
  --max-new-tokens 5996 \\
  --temperature 0.7 \\
  --top-k 4 \\
  --top-p 1.0 \\
  --seed 7 \\
  --tensor-parallel-size auto \\
  --batch-size 96 \\
  --max-model-len 6144 \\
  --max-num-batched-tokens 16384 \\
  --gpu-memory-utilization 0.91 \\
  --optimization-level 2 \\
  --performance-mode balanced \\
  --async-scheduling \\
  --output-file recipes/evo2_phage_gen/data/checkpoints/generation/phix174_prompt4_temp0.7.jsonl

evo2_phage_generation jsonl-to-fasta \\
  --input-jsonl recipes/evo2_phage_gen/data/checkpoints/generation/phix174_prompt4_temp0.7.jsonl \\
  --output-fasta recipes/evo2_phage_gen/data/checkpoints/generation/phix174_prompt4_temp0.7.fasta
"""

print(prompt_sweep_example)

## Mini Prompt-File Smoke Result

A small prompt-file smoke validates the generation handoff without running the full 4-6 kb sweep:

```bash
evo2_phage_generation write-prompts \
  --output-dir recipes/evo2_phage_gen/data/checkpoints/generation/mini_smoke/prompts \
  --prompt-lengths 4 \
  --num-prompts 2 \
  --id-prefix mini

RL_CHECKPOINT=recipes/evo2_phage_gen/data/checkpoints/evo2_7b_microviridae_mbridge
VLLM_EXPORT=recipes/evo2_phage_gen/data/checkpoints/evo2_7b_microviridae_vllm
TOKENIZER_JSON=recipes/evo2_megatron/tokenizers/nucleotide_fast_tokenizer_512/tokenizer.json
evo2_export_mbridge_to_vllm "$RL_CHECKPOINT" "$VLLM_EXPORT" --max-shard-size 2GiB

infer_evo2 \
  --model "$VLLM_EXPORT" \
  --rl-checkpoint "$RL_CHECKPOINT" \
  --rl-tokenizer-json "$TOKENIZER_JSON" \
  --prompt-file recipes/evo2_phage_gen/data/checkpoints/generation/mini_smoke/prompts/mini_prompt4_2.jsonl \
  --max-new-tokens 16 \
  --temperature 0.7 \
  --top-k 4 \
  --top-p 1.0 \
  --seed 11 \
  --tensor-parallel-size auto \
  --batch-size 2 \
  --max-model-len 128 \
  --max-num-batched-tokens 16384 \
  --gpu-memory-utilization 0.91 \
  --optimization-level 2 \
  --performance-mode balanced \
  --async-scheduling \
  --output-file recipes/evo2_phage_gen/data/checkpoints/generation/mini_smoke/mini_prompt4_temp0.7.jsonl
```

Validate that both rows contain exactly 16 completion token IDs, aligned finite chosen-token log probabilities, and no EOS. JSONL-to-FASTA reconstruction strips the `+~` soft tokens and prepends the `GAGT` nucleotide prefix. This short smoke checks the export and vLLM handoff; it does not replace the 4-6 kb generation and QC gate.

## Arc External QC Config

The Arc filtering pipeline lives in `recipes/evo2_phage_gen/data/external/arc_evo2/phage_gen/pipelines`. This recipe adds `configs/arc_genome_design_filtering_local.yaml` for local prompt-sweep FASTAs and `configs/arc_genome_design_filtering_curated_smoke.yaml` for Arc's bundled `all_generated_phages.fasta`. Enable external stages only after their tool/database paths are populated.

Prepare a patched workdir first. This copies Arc's pipeline files and rewrites three user-local assumptions: the import-time PhiX174 FASTA path in `genetic_architecture.py`, the hard-coded Prodigal binary path, and the hard-coded CheckV database path. The patched workdir uses the repo-local PhiX174 reference, `prodigal` from `PATH`, and the caller's `CHECKVDB` environment.

```bash
evo2_phage_prepare_arc_pipeline --overwrite
```

Check prerequisites. Use `--warn-only` before generation has produced the configured prompt-sweep FASTA; drop it when using the checker as a hard gate.

```bash
evo2_phage_check_external_qc \
  --config recipes/evo2_phage_gen/configs/arc_genome_design_filtering_curated_smoke.yaml \
  --genetic-architecture-import-fasta recipes/evo2_phage_gen/data/external/arc_evo2/phage_gen/data/NC_001422_1.fna \
  --warn-only
```

Run Arc's pipeline from the repo root while executing the patched script path so sibling imports resolve:

```bash
python recipes/evo2_phage_gen/data/arc_pipeline_patched/genome_design_filtering_pipeline.py \
  recipes/evo2_phage_gen/configs/arc_genome_design_filtering_curated_smoke.yaml
```

Patched curated smoke result: Arc's nucleotide-only pipeline loads 302 bundled curated candidates and retains 302 after valid DNA characters, 4-6 kb length, 30-65 percent GC, and nucleotide homopolymer length at most 10. This FASTA is already curated, so it is not the denominator for the raw-generation ~10% pass-rate replication.

The recipe environment installs the Python-facing QC tools. Use `evo2_phage_prepare_external_assets` for the Prodigal wrapper, MMseqs2-GPU binary, PHROGs annotation, and GPU-padded PHROGs sequence DB. CheckV additionally requires `diamond` on `PATH` before its database build can complete.

Before enabling the full external cascade, also fetch any required Zenodo raw SFT FASTA and build the PhiX174 G-protein MMseqs database under `recipes/evo2_phage_gen/data/external/` as described in the README.

## Step 4 - Nucleotide QC Smoke

This reproduces the dependency-light nucleotide layer of Arc's filtering pipeline on the bundled generated phage FASTA. It checks valid DNA alphabet, 4-6 kb length, 30-65 percent GC, and nucleotide homopolymer length at most 10.

```bash
evo2_phage_nucleotide_qc \
  --input-fasta recipes/evo2_phage_gen/data/external/arc_evo2/phage_gen/data/all_generated_phages.fasta \
  --output-dir recipes/evo2_phage_gen/data/checkpoints/phage_qc_smoke
```

Expected smoke result on the curated paper candidates: 302 initial sequences and 302 pass all dependency-light nucleotide filters.

In [ ]:
qc_cmd = [
    "bash",
    "-lc",
    "source recipes/evo2_phage_gen/.ci_test_env.sh && "
    "evo2_phage_nucleotide_qc "
    "--input-fasta recipes/evo2_phage_gen/data/external/arc_evo2/phage_gen/data/all_generated_phages.fasta "
    "--output-dir recipes/evo2_phage_gen/data/checkpoints/phage_qc_smoke",
]
subprocess.run(qc_cmd, cwd=REPO, check=True)

counts_path = CHECKPOINTS / "phage_qc_smoke" / "qc2_nt_filter_counts.csv"
print(counts_path.read_text())

## Step 5 - Online Reward Smoke

The first online reward is intentionally cheap enough to run during RL rollouts. It wraps the nucleotide metrics as scalar reward components and writes per-sequence diagnostics.

```bash
evo2_phage_score_fasta \
  --input-fasta recipes/evo2_phage_gen/data/external/arc_evo2/phage_gen/data/all_generated_phages.fasta \
  --output-csv recipes/evo2_phage_gen/data/checkpoints/phage_qc_smoke/rewards.csv
```

Expected smoke result on Arc's curated FASTA: 302 rows with mean/min/max reward all equal to 1.0.

In [ ]:
reward_cmd = [
    "bash",
    "-lc",
    "source recipes/evo2_phage_gen/.ci_test_env.sh && "
    "evo2_phage_score_fasta "
    "--input-fasta recipes/evo2_phage_gen/data/external/arc_evo2/phage_gen/data/all_generated_phages.fasta "
    "--output-csv recipes/evo2_phage_gen/data/checkpoints/phage_qc_smoke/rewards.csv",
]
subprocess.run(reward_cmd, cwd=REPO, check=True)

reward_path = CHECKPOINTS / "phage_qc_smoke" / "rewards.csv"
with reward_path.open() as f:
    reward_values = [float(row["reward"]) for row in csv.DictReader(f)]

{
    "count": len(reward_values),
    "mean": sum(reward_values) / len(reward_values),
    "min": min(reward_values),
    "max": max(reward_values),
}

## Step 6 - vLLM Inference Smoke

After the Microviridae checkpoint is converted, export it and run a short generation smoke before full 4-6 kb generation. This confirms the exact checkpoint loads through the public vLLM path.

```bash
mkdir -p recipes/evo2_phage_gen/data/checkpoints/generation
RL_CHECKPOINT=recipes/evo2_phage_gen/data/checkpoints/evo2_7b_microviridae_mbridge
VLLM_EXPORT=recipes/evo2_phage_gen/data/checkpoints/evo2_7b_microviridae_vllm
TOKENIZER_JSON=recipes/evo2_megatron/tokenizers/nucleotide_fast_tokenizer_512/tokenizer.json
evo2_export_mbridge_to_vllm "$RL_CHECKPOINT" "$VLLM_EXPORT" --max-shard-size 2GiB

infer_evo2 \
  --model "$VLLM_EXPORT" \
  --rl-checkpoint "$RL_CHECKPOINT" \
  --rl-tokenizer-json "$TOKENIZER_JSON" \
  --prompt GAGTTTTATCGCTTCCATGACGCAGAAGTTAACACTTTCGGATATTTCTGATGAGTCGAAAAATTATCTT \
  --max-new-tokens 8 \
  --temperature 0.8 \
  --top-k 4 \
  --seed 7 \
  --tensor-parallel-size auto \
  --batch-size 1 \
  --max-model-len 128 \
  --max-num-batched-tokens 16384 \
  --gpu-memory-utilization 0.91 \
  --optimization-level 2 \
  --performance-mode balanced \
  --async-scheduling \
  --output-file recipes/evo2_phage_gen/data/checkpoints/generation/smoke_microviridae.jsonl
```

Require an exact eight-token completion with aligned finite chosen-token log probabilities and no EOS. This smoke validates loading and output shape; use the mixed-length B96 qualification artifact for accuracy and performance claims.

## Step 7 - Prior Analysis For Ambiguous Filters

The converter uses a data-driven balanced prior for the ambiguous long-filter inverse. Re-run this analysis when changing the inverse or validating a new Evo2 checkpoint family.

```bash
evo2_analyze_inverse_prior \
  --checkpoint-dir "$HOME/.cache/bionemo/d663c529ac7ae0b6f2fd3a852253a484bd8a6576992e9ec73045ce7af2365990-nemo2_evo2_1b_8k.tar.gz.untar" \
  --output-json recipes/evo2_phage_gen/data/checkpoints/prior_analysis/evo2_1b_8k_prior.json

evo2_analyze_inverse_prior \
  --checkpoint-dir "$HOME/.cache/bionemo/78fc05536e1a9bd2febacea079a4beedf93ddcba1c69ac24690a5f7b649a0655-nemo2_evo2_7b_8k.tar.gz.untar" \
  --output-json recipes/evo2_phage_gen/data/checkpoints/prior_analysis/evo2_7b_8k_prior.json
```

Observed result: trained `p` and `gamma` are centered near zero and track each other, while fewer than 0.01 percent of `gamma` values remain inside the original log-init support.

## Step 8 - RL Scaffold

The GRPO scaffold registers `phage_qc` as a NeMo-RL environment and configures generation through Megatron rather than vLLM. After building the recipe environment, run the readiness checker before launching GRPO:

```bash
cd recipes/evo2_phage_gen
evo2_phage_check_rl --warn-only

evo2_phage_run_grpo \
  --config configs/grpo_phage_megatron.yaml
cd ../..
```

Current readiness result: NeMo-RL imports, the local recipe GRPO launcher, copied GRPO defaults, converted checkpoint, checkpoint `run_config.yaml`, BioNeMo checkpoint targets, tokenizer, prompt data, Megatron generation backend, inherited colocated Megatron GRPO topology, `phage_qc` environment config, and 2-GPU requirement pass on this workspace. Environment setup repairs upstream NeMo-RL package discovery and applies the Evo2/MBridge patch.

## Exercises And Extensions

1. Change one nucleotide reward threshold, re-run the reward smoke, and explain which component changed.
2. Run the CI converter test with an empty cache and record the download/runtime overhead.
3. Once NeMo-RL dependencies are installed, dry-run `evo2_phage_run_grpo --help` and then inspect the resolved config.
4. Add one offline QC component, such as ORF count or protein hit count, as a batched reward diagnostic rather than an online rollout reward.

Common pitfall: do not use the `evo2_7b_microviridae` model-size name. The released Microviridae checkpoint is a fine-tune of the 7B base architecture, so use `--model-size evo2_7b_base --seq-length 10240`.